# Bab 14 · Regresi Linear dan Gradient Descent dari Nol

**Notebook praktikum mahasiswa**  
Versi 2.0 · Pemrograman Komputer

- Menggunakan konvensi MSE secara konsisten.
- Memeriksa gradien sebelum melatih model.
- Menilai konvergensi dan peran standardisasi fitur.

### Petunjuk menjalankan sel · versi 2.0

Jalankan sel berurutan dari atas ke bawah. Setiap fungsi mandiri diletakkan pada sel tersendiri; sel pemanggilan atau pengujiannya menyusul setelah definisi. Setelah menyunting fungsi, jalankan ulang sel definisinya, lalu sel pengujiannya.

Sel persiapan dan fungsi pemeriksa cukup dijalankan; bagian yang Anda kerjakan ditandai **[ISI KODE]**. Metode yang membentuk satu kelas serta fungsi bersarang tetap disatukan karena merupakan satu kesatuan Python.

## Alur praktikum

**Duga → Jalankan → Selidiki → Isi kode → Periksa → Jelaskan**

Perkiraan waktu: 90–120 menit. Kerjakan berpasangan; tukar peran penulis kode dan pemeriksa setiap dua latihan.

| Penanda | Yang Anda kerjakan |
|---|---|
| [BACA] | Pahami konsep, kontrak fungsi, dan kasus batas. |
| [DUGA] | Tulis prediksi sebelum menjalankan contoh. |
| [COBA] | Jalankan contoh dan ubah satu hal untuk menyelidiki hasilnya. |
| [ISI KODE] | Lengkapi fungsi atau kelas; pertahankan nama dan parameternya. |
| [CEK OTOMATIS] | Jalankan pengujian yang terlihat, lalu gunakan pesannya untuk memperbaiki kode. |
| [REFLEKSI] | Jelaskan alasan dan bukti, bukan hanya menyalin keluaran. |

Impor file `.ipynb` ini ke notebook Python di Kaggle. Gunakan CPU; data kecil disediakan dalam notebook. Jalankan sel dari atas ke bawah. Pustaka yang diperlukan diimpor pada sel persiapan; tidak ada perintah instalasi atau unduhan.

`BELUM DIISI` adalah status normal pada notebook awal. Ganti `raise BelumDiisi()` dengan pekerjaan Anda. `LULUS` berarti memenuhi kasus uji yang tersedia, bukan bukti bahwa semua kemungkinan input sudah benar. Sel pengujian harus tetap utuh.

Jika kode berulang tanpa selesai, hentikan eksekusi, periksa batas perulangan, lalu jalankan ulang. Sebelum mengumpulkan, mulai ulang sesi Python dan jalankan seluruh sel agar hasil tidak bergantung pada variabel lama.

### Identitas

- Nama: …
- NIM: …
- Rekan diskusi: …
- Tanggal: …

**Persiapan dan pengaturan** · bagian 1 dari 9

In [ ]:
# [COBA] Jalankan sekali di awal; pemeriksaan tersedia untuk dibaca.
import math
import sys
from copy import deepcopy
from pathlib import Path
from tempfile import TemporaryDirectory

**Definisi `BelumDiisi`** · bagian 2 dari 9

In [ ]:
class BelumDiisi(Exception):
    """Penanda latihan yang belum dikerjakan."""

**Definisi `sama`** · bagian 3 dari 9

In [ ]:
def sama(aktual, harapan):
    assert aktual == harapan, f"Diharapkan {harapan!r}; diperoleh {aktual!r}"

**Definisi `dekat`** · bagian 4 dari 9

In [ ]:
def dekat(aktual, harapan, atol=1e-8, rtol=1e-7):
    assert math.isclose(
        aktual, harapan, abs_tol=atol, rel_tol=rtol
    ), f"Diharapkan sekitar {harapan!r}; diperoleh {aktual!r}"

**Definisi `harus_galat`** · bagian 5 dari 9

In [ ]:
def harus_galat(jenis, panggil):
    try:
        panggil()
    except BelumDiisi:
        raise
    except jenis:
        return
    raise AssertionError(f"Seharusnya memunculkan {jenis.__name__}")

**Persiapan dan pengaturan** · bagian 6 dari 9

In [ ]:
DAFTAR_UJI = {}

**Definisi `cek`** · bagian 7 dari 9

In [ ]:
def cek(nomor, fungsi_uji, tampil=True):
    DAFTAR_UJI[nomor] = fungsi_uji
    try:
        fungsi_uji()
        status, pesan = "LULUS", "Semua kasus uji pada latihan ini sesuai."
    except BelumDiisi:
        status, pesan = (
            "BELUM DIISI",
            "Lengkapi sel [ISI KODE], jalankan, lalu ulangi pemeriksaan.",
        )
    except AssertionError as err:
        status, pesan = (
            "PERLU PERBAIKAN",
            str(err) or "Hasil belum sesuai kontrak latihan.",
        )
    except Exception as err:
        status, pesan = "GALAT", f"{type(err).__name__}: {err}"
    if tampil:
        print(f"Latihan {nomor} | {status}\n{pesan}")
    return status

**Definisi `rekap`** · bagian 8 dari 9

In [ ]:
def rekap():
    # Uji ulang fungsi terkini agar rekap tidak memakai status lama.
    hasil = {
        nomor: cek(nomor, uji, tampil=False)
        for nomor, uji in sorted(DAFTAR_UJI.items())
    }
    for nomor, status in hasil.items():
        print(f"  Latihan {nomor}: {status}")
    lulus = sum(s == "LULUS" for s in hasil.values())
    print(f"\nKemajuan uji otomatis: {lulus}/{JUMLAH_LATIHAN} latihan lulus.")
    print(
        "Refleksi, penjelasan, dan kualitas penyajian diperiksa bersama asisten."
    )
    return hasil

**Persiapan dan pengaturan** · bagian 9 dari 9

In [ ]:
print("Python:", sys.version.split()[0])
print("Siap. Jalankan notebook dari atas ke bawah.")
JUMLAH_LATIHAN = 4
import numpy as np

print("NumPy:", np.__version__)

## [BACA] Konsep inti

X adalah matriks desain; kolom bias sudah ditambahkan oleh pemanggil. Prediksi adalah Xw. Gunakan J(w)=||Xw−y||²/n (tanpa faktor 1/2), sehingga gradiennya 2Xᵀ(Xw−y)/n. Gradient descent memperbarui w dengan mengurangi laju belajar kali gradien. Standardisasi fitur tidak diterapkan pada kolom bias. Statistik standardisasi dipelajari hanya dari data latih.

## [DUGA] Prediksi sebelum eksekusi

Pada fungsi biaya w², laju mana yang membuat biaya membesar? Mengapa tanda pembaruan penting?

**Prediksi saya:** …

**Alasan:** …

In [ ]:
# [COBA]
w = 3.0
for alpha in [0.1, 1.1]:
    lintasan = [w]
    z = w
    for _ in range(8):
        z = z - alpha * 2 * z
        lintasan.append(z)
    print("alpha:", alpha, "biaya:", np.round(np.array(lintasan) ** 2, 3))

**[REFLEKSI]** Apa perbedaan prediksi dan hasil? Ubah satu input pada contoh, tulis hasilnya, lalu jelaskan konsep yang ditunjukkan.

**Jawaban:** …

## Latihan 1 · Fungsi biaya MSE

**[ISI KODE]**

Buat `biaya(X, y, w)` yang mengembalikan skalar MSE. X 2D, y dan w 1D dengan bentuk sesuai, semua berhingga, dan data tidak kosong. Gunakan rerata kuadrat galat, bukan jumlah atau akar galat.

> Petunjuk: Hitung satu vektor residu, lalu rata-ratakan kuadrat elemennya.

In [ ]:
# [ISI KODE]
def biaya(X, y, w):
    raise BelumDiisi()

**Definisi `uji_01`** · bagian 1 dari 2

In [ ]:
# [CEK OTOMATIS] Kasus uji terbuka; jalankan setelah sel jawaban.
def uji_01():
    X = np.array([[1.0, 0.0], [1.0, 1.0]])
    y = np.array([1.0, 3.0])
    dekat(biaya(X, y, np.zeros(2)), 5)
    dekat(biaya(X, y, np.array([1.0, 2.0])), 0)
    dekat(biaya(np.vstack([X, X]), np.tile(y, 2), np.zeros(2)), 5)

**Jalankan pemeriksaan** · bagian 2 dari 2

In [ ]:
cek(1, uji_01)

## Latihan 2 · Gradien yang sesuai biaya

**[ISI KODE]**

Buat `gradien(X, y, w)` untuk MSE pada latihan sebelumnya. Kembalikan vektor berbentuk sama dengan w. Jangan mengubah w. Tes menghitung beda pusat secara mandiri agar tidak bergantung pada jawaban latihan 1.

> Petunjuk: Perhatikan transpos X serta bentuk residu.

In [ ]:
# [ISI KODE]
def gradien(X, y, w):
    raise BelumDiisi()

**Definisi `uji_02`** · bagian 1 dari 2

In [ ]:
# [CEK OTOMATIS] Kasus uji terbuka; jalankan setelah sel jawaban.
def uji_02():
    X = np.array([[1.0, 0.0], [1.0, 1.0], [1.0, 2.0]])
    y = np.array([1.0, 3.0, 5.0])
    w = np.array([0.3, -0.4])
    awal = w.copy()
    g = gradien(X, y, w)
    sama(g.shape, w.shape)
    eps = 1e-5

    def J(v):
        return np.mean((X @ v - y) ** 2)

    for j in range(len(w)):
        d = np.zeros_like(w)
        d[j] = eps
        dekat(g[j], (J(w + d) - J(w - d)) / (2 * eps), atol=1e-7)
    np.testing.assert_array_equal(w, awal)
    np.testing.assert_allclose(
        gradien(X, y, np.array([1.0, 2.0])), [0, 0], atol=1e-12
    )

**Jalankan pemeriksaan** · bagian 2 dari 2

In [ ]:
cek(2, uji_02)

## Latihan 3 · Loop pelatihan dengan riwayat

**[ISI KODE]**

Buat `latih_gd(X, y, alpha=0.05, iterasi=500)` dari bobot nol. Kembalikan `(w, riwayat)`, riwayat berupa array biaya awal dan setelah setiap pembaruan, panjang iterasi+1. Tolak alpha ≤ 0 atau iterasi < 0 dengan ValueError. iterasi bilangan bulat. Gunakan fungsi biaya/gradien yang sudah dikerjakan atau tulis perhitungannya di fungsi ini. Uji memakai data kecil dan alpha stabil; riwayat yang meningkat harus tetap terlihat.

> Petunjuk: Catat biaya sebelum loop dan sesudah setiap pembaruan.

In [ ]:
# [ISI KODE]
def latih_gd(X, y, alpha=0.05, iterasi=500):
    raise BelumDiisi()

**Definisi `uji_03`** · bagian 1 dari 2

In [ ]:
# [CEK OTOMATIS] Kasus uji terbuka; jalankan setelah sel jawaban.
def uji_03():
    x = np.linspace(-1, 1, 21)
    X = np.column_stack([np.ones(len(x)), x])
    y = 1 + 2 * x
    w, h = latih_gd(X, y, alpha=0.1, iterasi=500)
    sama(len(h), 501)
    np.testing.assert_allclose(w, [1, 2], atol=1e-6)
    dekat(h[0], np.mean(y * y))
    dekat(h[-1], np.mean((X @ w - y) ** 2), atol=1e-12)
    assert np.all(
        np.diff(h) <= 1e-10
    ), "Biaya seharusnya turun untuk kasus stabil ini."
    w0, h0 = latih_gd(X, y, iterasi=0)
    np.testing.assert_array_equal(w0, [0, 0])
    sama(len(h0), 1)
    harus_galat(ValueError, lambda: latih_gd(X, y, alpha=0))
    harus_galat(ValueError, lambda: latih_gd(X, y, iterasi=-1))

**Jalankan pemeriksaan** · bagian 2 dari 2

In [ ]:
cek(3, uji_03)

## Latihan 4 · Standardisasi latih dan uji

**[ISI KODE]**

Buat `skala_latih_uji(latih, uji)` untuk array float 2D, jumlah kolom sama, latih tidak kosong. Kembalikan `(Z_latih, Z_uji, mu, skala)`. Pelajari mu dan sd populasi dari latih; jika sd kolom nol, gunakan skala=1. Terapkan mu dan skala yang sama ke uji. Kedua input tidak memiliki kolom bias dan tidak boleh berubah.

> Petunjuk: Jangan menghitung rerata gabungan atau rerata data uji.

In [ ]:
# [ISI KODE]
def skala_latih_uji(latih, uji):
    raise BelumDiisi()

**Definisi `uji_04`** · bagian 1 dari 2

In [ ]:
# [CEK OTOMATIS] Kasus uji terbuka; jalankan setelah sel jawaban.
def uji_04():
    lat = np.array([[1.0, 7.0], [3.0, 7.0]])
    uji = np.array([[101.0, 9.0]])
    awal = lat.copy()
    zl, zu, mu, s = skala_latih_uji(lat, uji)
    np.testing.assert_allclose(mu, [2, 7])
    np.testing.assert_allclose(s, [1, 1])
    np.testing.assert_allclose(zl, [[-1, 0], [1, 0]])
    np.testing.assert_allclose(zu, [[99, 2]])
    np.testing.assert_array_equal(lat, awal)
    np.testing.assert_array_equal(uji, [[101, 9]])

**Jalankan pemeriksaan** · bagian 2 dari 2

In [ ]:
cek(4, uji_04)

## [REFLEKSI] Refleksi akhir

Bagian ini membantu Anda merangkum pemahaman, mengenali kesulitan, dan menjelaskan alasan di balik kode. Tulis jawaban singkat berdasarkan percobaan Anda; bukan sekadar menyalin keluaran.

1. Pilih satu latihan. Jelaskan alur kode Anda dengan satu contoh input dan hasilnya.
2. Tuliskan satu kesalahan yang sempat terjadi, penyebabnya, dan cara memperbaikinya.
3. Usulkan satu kasus uji tambahan yang belum tercakup. Nyatakan hasil yang Anda harapkan dan alasannya.
4. Apa batas kesimpulan yang boleh dibuat dari hasil praktikum ini?

**Jawaban:** …

### Tantangan pengembangan

Tambahkan kasus uji usulan Anda pada sel di bawah. Pastikan kasus tersebut bisa membedakan implementasi benar dan satu kesalahan yang masuk akal. Diskusikan dengan asisten sebelum mengubah kontrak fungsi.

In [ ]:
# [ISI KODE OPSIONAL] Tambahkan eksperimen atau pengujian buatan Anda.
# Jelaskan harapan Anda pada komentar sebelum menjalankannya.

In [ ]:
# [CEK OTOMATIS] Uji ulang seluruh latihan yang sudah didaftarkan.
status_akhir = rekap()

## Sebelum mengumpulkan

- [ ] Identitas dan prediksi sudah diisi.
- [ ] Semua latihan sudah dikerjakan dan diperiksa dari sesi baru.
- [ ] refleksi akhir berisi penjelasan dengan bukti keluaran.
- [ ] Notebook disimpan dengan nama dan NIM; jangan hanya mengumpulkan HTML.

Rubrik diskusi: ketepatan kode 60%, penjelasan dan kasus batas 25%, keterbacaan serta kemampuan dijalankan ulang 15%. Rekap otomatis membantu belajar; penilaian akhir tetap memerlukan pemeriksaan asisten.

Rujukan: bab yang bersesuaian pada buku *Python untuk Machine Learning dan Data Science* dan modul praktikum. Latihan di notebook ini merupakan adaptasi terarah untuk praktikum, bukan seluruh soal akhir bab.